In [1]:
!which python3

/usr/bin/python3


In [1]:
import cv2
import time
import threading
from pathlib import Path
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

from jetbot import Camera, bgr8_to_jpeg
from PUTDriver import PUTDriver

Path('./dataset/').mkdir(exist_ok=True)

In [ ]:
config = {
    'model': { 'path': '' },
    'robot': {
        'max_speed': 0.40,   # bylo 0.22 - podnosimy aby zrobic miejsce na turbo
        'max_steering': 0.5,
        'differential': { 'left': 1.0, 'right': 0.995 }
    }
}
driver = PUTDriver(config=config)

# Bez try/except - jesli kamera padnie, chcemy zobaczyc pelny traceback,
# a nie tylko jednolinijkowy str(e). Wtedy w terminalu Jetbota:
#   sudo systemctl restart nvargus-daemon
camera = Camera.instance()

Kamera i silniki zainicjowane poprawnie!


In [3]:
controller = widgets.Controller(index=0)

# Bez podglady - traitlets dlink jest kosztowny (kazda klatka -> JPEG -> widget),
# a my chcemy pelne 10 Hz w petli sterujacej i przy nagrywaniu.
# Kamera dalej dziala w tle (camera.value), tylko nic nie rysujemy.

record_button = widgets.ToggleButton(description='Start/Stop Nagrywanie', button_style='danger')
count_widget = widgets.IntText(description='Zapisano klatek:', value=0)
debug_widget = widgets.HTML(value="<b>Czekam na dane z pada...</b>")

display(widgets.VBox([
    controller,
    widgets.HBox([record_button, count_widget]),
    debug_widget
]))

In [4]:
run_thread = True
recording_session = None
index = 0

DEAD_ZONE = 0.05
TURBO_BUTTON = 7         # RT (analogowy trigger) - 0.0..1.0

# Skalowanie forward: bez turbo cap = stara predkosc (0.22),
# z turbo na maks cap = nowy max_speed (0.40). Plynnie blendowane triggerem.
OLD_MAX = 0.20
NEW_MAX = 0.40
NORMAL_SCALE = OLD_MAX / NEW_MAX   # ~0.55

def save_data(key, idx, image, forward, left):
    """Zapis jednej klatki do dataset/<key>/ + dopisanie wiersza do dataset/<key>.csv.
    Struktura zgodna z dataset_example: katalog z 0000.jpg, 0001.jpg, ... + CSV idx,forward,left.
    Zapisujemy ZESKALOWANY forward + skorygowany left (to co realnie dostaje sterownik).
    """
    if image is None:
        return False
    with open(f'./dataset/{key}.csv', 'a') as f:
        f.write(f'{idx},{forward},{left}\n')
    img_resized = cv2.resize(image, (224, 224))
    cv2.imwrite(f'./dataset/{key}/{idx:04d}.jpg', img_resized)
    return True

def driving_loop():
    """Sterowanie video-game-like + analogowe turbo na trigger 7:
    - axes: aktualny stan pada -> bezposrednio na silniki (bez historii, bez smoothing)
    - trigger 7 (RT): 0.0 = normalna jazda (cap 0.22), 1.0 = pelny gaz (cap 0.40)
    - na biegu wstecznym left jest odwracany (jak w grach: stick-prawo = zawsze trajektoria w prawo)
    - przy nagrywaniu lecimy zeskalowany forward + skorygowany left (zgodne ze sterownikiem)
    """
    global index, recording_session

    while run_thread:
        # 1. Aktualny stan pada (galki + turbo)
        try:
            forward_raw = -controller.axes[1].value
            left_raw = -0.6*controller.axes[2].value
        except IndexError:
            forward_raw = 0.0
            left_raw = 0.0

        try:
            nitro = controller.buttons[TURBO_BUTTON].value   # analog 0..1
        except IndexError:
            nitro = 0.0

        # 2. Martwa strefa
        forward_input = forward_raw if abs(forward_raw) > DEAD_ZONE else 0.0
        left_input = left_raw if abs(left_raw) > DEAD_ZONE else 0.0

        # 3. Skalowanie forward turbo'em - linear blend miedzy NORMAL_SCALE a 1.0
        scale = NORMAL_SCALE + (1.0 - NORMAL_SCALE) * nitro
        forward = forward_input * scale

        # 4. Inwersja steeru na biegu wstecznym (video-game convention)
        left = -left_input if forward < 0 else left_input

        # 5. Bezposrednio na silniki - bez smoothing, bez akumulacji.
        driver.update(forward, left)

        debug_info = (
            f"Naped -> Przod: {forward:.2f} (raw: {forward_input:.2f}, turbo: {nitro:.2f}) "
            f"| L/P: {left:.2f} (raw: {left_input:.2f}) | osi: {len(controller.axes)}"
        )

        # 6. Logika nagrywania (zapis lokalny zgodny z dataset_example)
        if record_button.value:
            if recording_session is None:
                recording_session = str(time.time())
                Path(f'./dataset/{recording_session}/').mkdir(exist_ok=True)
                index = 0

            if save_data(recording_session, index, camera.value, forward, left):
                index += 1
                count_widget.value = index
            debug_widget.value = (
                f"<b style='color:red;'>NAGRYWANIE: ./dataset/{recording_session}/</b>"
                f"<br>{debug_info}"
            )
        else:
            if recording_session is not None:
                recording_session = None
            debug_widget.value = f"<b>{debug_info}</b>"

        time.sleep(0.1)

thread = threading.Thread(target=driving_loop)
thread.start()

In [5]:
run_thread = False
thread.join()
driver.update(0.0, 0.0)
camera.stop()
debug_widget.value = "<b>Robot i kamera bezpiecznie wylaczone.</b>"